# Gold Layer: Customer Dimension (SCD Type 2)

**Pattern**: Slowly Changing Dimension Type 2  
**Tracks**: Full history of customer attribute changes

**Source Tables**:
- `silver.crm_customers_cdc` (primary customer info)
- `silver.erp_customers_cdc` (birthdate, gender)
- `silver.erp_customer_location_cdc` (country)

**SCD Type 2 Columns**:
- `effective_from`: When this version became active
- `effective_to`: When this version was superseded (9999-12-31 for current)
- `is_current`: TRUE for active version, FALSE for historical

**Business Use Cases**:
- Track customer moves (NYC → LA)
- Analyze orders with customer's location at order time
- Report on customer demographics over time

**Example**: Customer moves
```
customer_key | customer_id | city | effective_from | effective_to  | is_current
1            | C123        | NYC  | 2023-01-01     | 2024-06-15    | FALSE
2            | C123        | LA   | 2024-06-15     | 9999-12-31    | TRUE
```

In [0]:
%sql
-- Build source data with SCD Type 2 columns
CREATE OR REPLACE TEMPORARY VIEW customer_source AS
SELECT 
  ci.customer_id,
  ci.customer_number,
  ci.first_name,
  ci.last_name,
  la.country,
  ci.marital_status,
  CASE 
    WHEN ci.gender <> "n/a" THEN ci.gender 
    ELSE coalesce(ca.gender, "n/a")
  END AS gender,
  ca.birth_date AS birthdate,
  ci.created_date AS create_date,
  
  -- Add hash for change detection (all attributes except keys)
  md5(concat_ws('|', 
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE WHEN ci.gender <> "n/a" THEN ci.gender ELSE coalesce(ca.gender, "n/a") END,
    cast(ca.birth_date as string),
    cast(ci.created_date as string)
  )) as attribute_hash,
  
  -- SCD Type 2 columns
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
  
FROM silver.crm_customers_cdc ci
LEFT JOIN silver.erp_customers_cdc ca 
  ON ci.customer_number = ca.customer_number
LEFT JOIN silver.erp_customer_location_cdc la
  ON ci.customer_number = la.customer_number;

SELECT COUNT(*) as source_customers FROM customer_source;

In [0]:
%sql
-- Preview source data with SCD columns
SELECT * FROM customer_source LIMIT 5;

# Writing Gold Table

## Two-Step SCD Type 2 Workflow

**Cell 5 - Initial Load** (Run once, safe to re-run):
- Creates table if it doesn't exist
- Loads all customers with `is_current = TRUE`
- **Idempotent**: Won't insert duplicates if table already has data

**Cell 6 - Incremental Updates** (Run for all future updates):
- Closes out changed customers (sets `effective_to = today`, `is_current = FALSE`)
- Inserts new versions for changed customers
- Inserts brand new customers
- **Preserves all history** - never deletes records

## Typical Usage
1. **First time**: Run Cell 2 → Cell 5
2. **After CDC updates**: Run Cell 2 → Cell 6
3. **Verify history**: Run Cell 8

In [0]:
%sql
-- Initial load: Create table and load if it doesn't exist
-- Safe to run multiple times (idempotent)

CREATE TABLE IF NOT EXISTS workspace.gold.dim_customers (
  customer_key BIGINT,
  customer_id STRING,
  customer_number STRING,
  first_name STRING,
  last_name STRING,
  country STRING,
  marital_status STRING,
  gender STRING,
  birthdate DATE,
  create_date DATE,
  attribute_hash STRING,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN
)
USING DELTA;

-- Only insert if table is empty (initial load)
INSERT INTO workspace.gold.dim_customers
SELECT 
  ROW_NUMBER() OVER (ORDER BY customer_number) as customer_key,
  customer_id,
  customer_number,
  first_name,
  last_name,
  country,
  marital_status,
  gender,
  birthdate,
  create_date,
  attribute_hash,
  effective_from,
  effective_to,
  is_current
FROM customer_source
WHERE NOT EXISTS (SELECT 1 FROM workspace.gold.dim_customers LIMIT 1);

-- Show results
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT customer_number) as unique_customers,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions
FROM workspace.gold.dim_customers;

In [0]:
%sql
-- Run this cell for incremental updates (after initial load)
-- SCD Type 2: Close out changed customers and insert new versions

-- Step 1: Close out customers whose attributes have changed
MERGE INTO workspace.gold.dim_customers as target
USING customer_source as source
ON target.customer_number = source.customer_number 
   AND target.is_current = TRUE
WHEN MATCHED AND target.attribute_hash <> source.attribute_hash
THEN UPDATE SET
  effective_to = current_date(),
  is_current = FALSE;

-- Step 2: Insert new versions for changed customers
INSERT INTO workspace.gold.dim_customers
SELECT 
  (SELECT COALESCE(MAX(customer_key), 0) FROM workspace.gold.dim_customers) + ROW_NUMBER() OVER (ORDER BY source.customer_number) as customer_key,
  source.customer_id,
  source.customer_number,
  source.first_name,
  source.last_name,
  source.country,
  source.marital_status,
  source.gender,
  source.birthdate,
  source.create_date,
  source.attribute_hash,
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
FROM customer_source source
INNER JOIN (
  SELECT customer_number 
  FROM workspace.gold.dim_customers
  WHERE effective_to = current_date()
) changed
ON source.customer_number = changed.customer_number;

-- Step 3: Insert brand new customers (not in target at all)
INSERT INTO workspace.gold.dim_customers
SELECT 
  (SELECT COALESCE(MAX(customer_key), 0) FROM workspace.gold.dim_customers) + ROW_NUMBER() OVER (ORDER BY source.customer_number) as customer_key,
  source.customer_id,
  source.customer_number,
  source.first_name,
  source.last_name,
  source.country,
  source.marital_status,
  source.gender,
  source.birthdate,
  source.create_date,
  source.attribute_hash,
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
FROM customer_source source
WHERE NOT EXISTS (
  SELECT 1 FROM workspace.gold.dim_customers target
  WHERE target.customer_number = source.customer_number
);

-- Show summary
SELECT 
  'Updated' as operation,
  COUNT(*) as customers
FROM workspace.gold.dim_customers
WHERE effective_to = current_date()
UNION ALL
SELECT 
  'Current Total',
  COUNT(DISTINCT customer_number)
FROM workspace.gold.dim_customers;

# Data Quality Framework

**Retail Analytics Best Practice #2**: Implement data quality checks with quarantine pattern

## Quality Dimensions
1. **Completeness**: Critical columns must not be null
2. **Validity**: Data types and ranges must be correct
3. **Consistency**: Referential integrity across dimensions
4. **Uniqueness**: No duplicate business keys in current versions

## Quarantine Pattern
- Failed records → `gold.dim_customers_quarantine`
- Track failure reason, timestamp, and original data
- Investigate and remediate without blocking pipeline

## Quality Checks
- Run after Cell 6 (incremental update)
- Log metrics to `gold.data_quality_metrics`
- Alert on quality degradation

In [0]:
%sql
-- Verify SCD Type 2 implementation
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT customer_number) as unique_customers,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
  SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) as historical_versions,
  COUNT(*) - COUNT(DISTINCT customer_number) as customers_with_history
FROM workspace.gold.dim_customers;

-- Show customers with multiple versions (history tracking)
SELECT 
  customer_number,
  customer_key,
  first_name,
  last_name,
  country,
  effective_from,
  effective_to,
  is_current
FROM workspace.gold.dim_customers
WHERE customer_number IN (
  SELECT customer_number 
  FROM workspace.gold.dim_customers
  GROUP BY customer_number
  HAVING COUNT(*) > 1
)
ORDER BY customer_number, effective_from
LIMIT 20;